# Pipeline 05: Interactive UI & SLM Data Flywheel

**Goal:** Wrap the End-to-End Inference Engine into an interactive web application while establishing a Human-in-the-Loop (HITL) data collection pipeline.

**Architecture:**
This pipeline utilizes `Gradio` to render a front-end interface directly within the notebook. It serves two primary functions:
1. **Interactive Inference:** Allows users to dynamically select diverse out-of-distribution (OOD) images, input natural language overrides, and instantly visualize the deterministic mathematical rendering.
2. **Knowledge Distillation Logging:** Outputs the raw JSON schema generated by the "Teacher" LLM (Claude). Upon user verification of the visual result, a persistent logging mechanism appends the `[Natural Language Prompt -> Structured JSON]` pair to a `.jsonl` dataset. This dataset acts as the foundation for future parameter-efficient fine-tuning (PEFT) of smaller, edge-deployable open-source models.

In [1]:
# --- 1. Setup, Dependencies, and Diverse Image Sourcing ---
from google.colab import drive
drive.mount('/content/drive')

!pip install gradio anthropic segment-anything opencv-python-headless kornia -q

import os
import cv2
import json
import torch
import kornia
import urllib
import numpy as np
import gradio as gr
from PIL import Image
from google.colab import userdata
from anthropic import Anthropic
from segment_anything import sam_model_registry, SamAutomaticMaskGenerator

PROJECT = '/content/drive/MyDrive/photo-style-rl'
PROFILE_PATH = f'{PROJECT}/checkpoints/simon_advanced_color_profile.json'
DATASET_PATH = f'{PROJECT}/data/slm_training_dataset.jsonl'
SAM_CHECKPOINT = f'{PROJECT}/checkpoints/sam_vit_h_4b8939.pth'

# Ensure data directory exists
os.makedirs(os.path.dirname(DATASET_PATH), exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
client = Anthropic(api_key=userdata.get('ANTHROPIC_API_KEY'))

# Download Diverse OOD Test Images
test_images = {
    "Portrait": "https://images.unsplash.com/photo-1534528741775-53994a69daeb?q=80&w=1024",
    "Moody Street": "https://images.unsplash.com/photo-1555899434-94d1368aa7af?q=80&w=1024",
    "Architecture": "https://images.unsplash.com/photo-1517404215738-15263e9f9178?q=80&w=1024",
    "Landscape": "https://images.unsplash.com/photo-1464822759023-fed622ff2c3b?q=80&w=1024",
    "Food Macro": "https://images.unsplash.com/photo-1550547660-d9450f859349?q=80&w=1024"
}

IMAGE_CACHE = {}
for name, url in test_images.items():
    safe_name = f"test_{name.replace(' ', '_').lower()}.jpg"
    if not os.path.exists(safe_name):
        urllib.request.urlretrieve(url, safe_name)
    IMAGE_CACHE[name] = safe_name

# Load Base Profile
with open(PROFILE_PATH, 'r') as f:
    style_profile = json.load(f)

print(f"Dependencies loaded. Persistent training log located at: {DATASET_PATH}")

Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 469.4/469.4 kB 15.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 57.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 114.1 MB/s eta 0:00:00
Dependencies loaded. Persistent training log located at: /content/drive/MyDrive/photo-style-rl/data/slm_training_dataset.jsonl


## The Application Backend

This section integrates the deterministic renderer and the LLM parsing logic into backend functions optimized for Gradio. The core addition is the `save_to_dataset` function, which formats verified interactions into JSON Lines (`.jsonl`), the industry-standard format for causal language model fine-tuning.

In [2]:
# --- 2. Backend Logic & The Render Engine ---

# (We redefine the DeterministicRenderer here to ensure the Colab environment is self-contained)
class DeterministicRenderer:
    def __init__(self):
        self.hue_bins = {
            'reds': ((345, 360), (0, 15)), 'oranges': ((15, 45),), 'yellows': ((45, 75),),
            'greens': ((75, 135),), 'aquas': ((135, 195),), 'blues': ((195, 255),),
            'purples': ((255, 315),), 'magentas': ((315, 345),)
        }

    def apply_tone_curve(self, img_hsv, curve_params):
        if not curve_params: return img_hsv
        v_channel = img_hsv[:, :, 2]
        result_v = v_channel.copy()
        centers = {'blacks': 0.05, 'shadows': 0.225, 'midtones': 0.5, 'highlights': 0.775, 'whites': 0.95}
        for band, shift_pct in curve_params.items():
            if shift_pct == 0: continue
            shift_val = shift_pct / 100.0
            weight = np.exp(-((v_channel - centers[band]) ** 2) / (2 * 0.15 ** 2))
            result_v += weight * shift_val * v_channel * (1 - v_channel)
        img_hsv[:, :, 2] = np.clip(result_v, 0, 1)
        return img_hsv

    def apply_hsl_mixer(self, img_hsv, mixer_params):
        if not mixer_params: return img_hsv
        hue_deg = img_hsv[:, :, 0] * 360.0
        for color, shifts in mixer_params.items():
            if all(v == 0 for v in shifts.values()): continue
            mask = np.zeros_like(hue_deg, dtype=bool)
            for (low, high) in self.hue_bins[color]:
                mask |= (hue_deg >= low) & (hue_deg < high)
            if not np.any(mask): continue
            smooth_mask = cv2.GaussianBlur(mask.astype(np.float32), (5, 5), 0)
            if shifts.get('h', 0) != 0:
                img_hsv[:, :, 0] = (img_hsv[:, :, 0] + (shifts['h']/360.0)*smooth_mask) % 1.0
            if shifts.get('s', 0) != 0:
                img_hsv[:, :, 1] = np.clip(img_hsv[:, :, 1] * (1.0 + (shifts['s']/100.0)*smooth_mask), 0, 1)
            if shifts.get('l', 0) != 0:
                img_hsv[:, :, 2] = np.clip(img_hsv[:, :, 2] * (1.0 + (shifts['l']/100.0)*smooth_mask), 0, 1)
        return img_hsv

    def render_region(self, img_rgb, params):
        import matplotlib.colors as mcolors
        img_hsv = mcolors.rgb_to_hsv(img_rgb)
        img_hsv = self.apply_hsl_mixer(img_hsv, params.get('color_mixer', {}))
        img_hsv = self.apply_tone_curve(img_hsv, params.get('tone_curve', {}))
        return np.clip(mcolors.hsv_to_rgb(img_hsv), 0, 1)

renderer = DeterministicRenderer()

def process_edit(image_choice, prompt):
    """Parses prompt, extracts JSON, applies rendering."""
    if not prompt.strip():
        return IMAGE_CACHE[image_choice], "{}", "Please enter a prompt."

    # 1. Claude Translation
    sys_prompt = f"""You are translating photo editing instructions into JSON overrides.
    Available regions: {list(style_profile.keys())}
    Output ONLY a JSON dictionary where keys are region names (or 'global'), and values match this schema:
    {{"tone_curve": {{"blacks": pct, "shadows": pct, "midtones": pct, "highlights": pct, "whites": pct}},
    "color_mixer": {{"color_name": {{"h": deg, "s": pct, "l": pct}}}}}}"""

    try:
        response = client.messages.create(
            model="claude-sonnet-4-20250514", max_tokens=500,
            system=sys_prompt, messages=[{"role": "user", "content": prompt}]
        )
        import re
        json_str = re.search(r'\{.*\}', response.content[0].text, re.DOTALL).group()
        llm_overrides = json.loads(json_str)
        formatted_json = json.dumps(llm_overrides, indent=2)
    except Exception as e:
        return IMAGE_CACHE[image_choice], "{}", f"LLM Parsing Failed: {str(e)}"

    # 2. Rendering
    try:
        img = Image.open(IMAGE_CACHE[image_choice]).convert("RGB")
        img_np = np.array(img).astype(np.float32) / 255.0

        # Merge base profile (defaulting to subject/global for demo) with overrides
        import copy
        base_params = copy.deepcopy(style_profile.get('subject', {}))
        override = llm_overrides.get('subject', llm_overrides.get('global', {}))

        if 'tone_curve' in override:
            if 'tone_curve' not in base_params: base_params['tone_curve'] = {}
            base_params['tone_curve'].update(override['tone_curve'])
        if 'color_mixer' in override:
            if 'color_mixer' not in base_params: base_params['color_mixer'] = {}
            for color, shifts in override['color_mixer'].items():
                if color not in base_params['color_mixer']: base_params['color_mixer'][color] = {}
                base_params['color_mixer'][color].update(shifts)

        result_np = renderer.render_region(img_np.copy(), base_params)
        result_img = Image.fromarray((np.clip(result_np, 0, 1) * 255).astype(np.uint8))

        return result_img, formatted_json, "✅ Render Complete! Review and save if accurate."
    except Exception as e:
        return IMAGE_CACHE[image_choice], formatted_json, f"Rendering Failed: {str(e)}"

def save_to_dataset(prompt, json_output):
    """Appends verified interactions to the persistent JSONL file."""
    if not prompt or json_output == "{}" or not json_output:
        return "⚠️ Cannot save empty data."

    try:
        # Validate that the output is actually JSON before saving
        parsed_json = json.loads(json_output)

        # Format for instruction fine-tuning
        training_record = {
            "messages": [
                {"role": "user", "content": prompt},
                {"role": "assistant", "content": json.dumps(parsed_json)}
            ]
        }

        with open(DATASET_PATH, "a") as f:
            f.write(json.dumps(training_record) + "\n")

        return f"💾 Saved successfully! Total records in dataset: {sum(1 for _ in open(DATASET_PATH))}"
    except Exception as e:
        return f"❌ Error saving: {str(e)}"

## Gradio Application Launch

Execute the cell below to launch the interactive front-end.
1. Select a diverse test image.
2. Type a natural language editing instruction.
3. Observe the generated JSON and the artifact-free mathematical render.
4. If the LLM successfully interpreted your intent, click **"💾 Save to Training Dataset"** to persist the interaction for future SLM distillation.

In [3]:
# --- 3. Gradio Web UI ---
with gr.Blocks(theme=gr.themes.Monochrome()) as app:
    gr.Markdown("# 🎛️ Advanced Color Science Engine & Data Flywheel")
    gr.Markdown("Test the inference pipeline on OOD images and log successful translations for SLM Distillation.")

    with gr.Row():
        with gr.Column(scale=1):
            image_selector = gr.Radio(
                choices=list(test_images.keys()),
                value="Portrait",
                label="Select Test Subject"
            )
            prompt_input = gr.Textbox(
                lines=3,
                placeholder="e.g., Apply my style, but drastically desaturate the blues and warm up the highlights...",
                label="Natural Language Instruction"
            )
            generate_btn = gr.Button("Generate Render", variant="primary")

            gr.Markdown("### Teacher LLM Output (Claude)")
            json_output = gr.Code(language="json", label="Generated Override Weights", lines=15)

            save_btn = gr.Button("💾 Save to Training Dataset", variant="secondary")
            status_text = gr.Markdown()

        with gr.Column(scale=1):
            image_output = gr.Image(label="Rendered Result", type="pil", interactive=False)

    # Event Listeners
    generate_btn.click(
        fn=process_edit,
        inputs=[image_selector, prompt_input],
        outputs=[image_output, json_output, status_text]
    )

    save_btn.click(
        fn=save_to_dataset,
        inputs=[prompt_input, json_output],
        outputs=[status_text]
    )

# Launch the app with a public share link
app.launch(share=True, debug=False)

/tmp/ipykernel_3864/2201603354.py:2: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Monochrome()) as app:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://31d561489893e299ee.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [6]:
import os

# --- Utility: Undo Last Saved Record ---
DATASET_PATH = f'{PROJECT}/data/slm_training_dataset.jsonl'

if os.path.exists(DATASET_PATH):
    with open(DATASET_PATH, 'r') as f:
        lines = f.readlines()

    if len(lines) > 0:
        # Remove the last line
        deleted_record = lines.pop()

        # Overwrite the file with the remaining lines
        with open(DATASET_PATH, 'w') as f:
            f.writelines(lines)

        print(f"✅ Successfully deleted the last record.")
        print(f"🗑️ Deleted Data: {deleted_record[:100]}...")
        print(f"📊 Total records remaining: {len(lines)}")
    else:
        print("⚠️ The dataset is already empty.")
else:
    print("⚠️ Dataset file not found.")

✅ Successfully deleted the last record.
🗑️ Deleted Data: {"messages": [{"role": "user", "content": "apply my style, but make the subject a bit brighter and m...
📊 Total records remaining: 0
